# Comparing movement-objective models

Estimates, for every subject and task condition, how strongly each of three
*basis* conditions contributes to the observed movement, and compares how well
different feature sets (spatiotemporal, joint kinematics, EMG) support that
decomposition.

Each basis condition isolates one task objective — walk fast
(`s2a0b0`), place the feet accurately (`s1a2b0`), stay balanced (`s1a0b2`).
Any other condition's feature vector is regressed onto the three basis vectors,
and the resulting coefficients are read as the weight the participant placed on
each objective. Coefficients are fit relative to the neutral condition
(`s1a0b0`) when `REGRESS_DIFF` is on.

**Inputs**

* `data/decomp_*.csv` — per-subject component scores from
  `00_reference_extract_movement_features.ipynb`. Edit `model_files` to choose which to compare.
* `data/data_BMH*.xlsx` — spatiotemporal metrics and energetics.
* `data/Subjective_Responses.xlsx` — participants' self-reported priorities.

**Outputs.** A goodness-of-fit table and paired tests across feature sets, the
estimated weights (`weights_df_*.xlsx`).

Run the notebook top to bottom from the repository root.

## Setup

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import os
import re
import colorsys
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import colors as mcolors
import seaborn as sns

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from scipy import stats
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multitest import multipletests

# Paths are relative to the repository root
DATA_DIR = Path("data")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

## Weight estimation

For one subject and one condition, the three basis conditions form the columns
of the design matrix and the target condition is the response; features (PCA/NMF
components, or raw gait metrics) are the observations. Fits are unconstrained
least squares, so coefficients may be negative — see the negative-weight
diagnostic below. An interaction model is fit alongside the additive one and
returned for comparison.

In [ ]:
def estimate_weights(
    final_df,
    features,
    basis_codes=("s2a0b0", "s1a2b0", "s1a0b2"),
    skip_subjects=(),
    degree=2,
    interaction_only=True,
    include_bias=False,
    null_code="s1a0b0",
    use_null_diff=False,   # if True, use (cond - null) and (basis - null)
):
    interaction_transformer = PolynomialFeatures(
        degree=degree,
        interaction_only=interaction_only,
        include_bias=include_bias,
    )
    model = LinearRegression()
    model_interacted = LinearRegression()

    rows = []
    rows_interacted = []
    y_comparison = []

    subjects = final_df["Subject"].unique()
    conditions = final_df["Condition"].unique()

    for subj in subjects:
        if subj in skip_subjects:
            continue

        subj_df = final_df[final_df["Subject"] == subj]

        # if we're using a null condition, make sure it exists
        if use_null_diff:
            if null_code is None:
                print(f"null_code must be provided when use_null_diff=True (Subject {subj})")
                continue
            null_df = subj_df[subj_df["Condition"] == null_code]
            if null_df.empty:
                print(f"null condition {null_code} does not exist for {subj}")
                continue
            null_row = null_df.iloc[0]
        else:
            null_row = None  # not used

        # ensure all basis codes exist for this subject
        if not all(code in subj_df["Condition"].values for code in basis_codes):
            print(f"basis code does not exist for {subj}")
            continue

        for cond in conditions:
            if use_null_diff and cond == null_code:
                continue
            
            subset = subj_df[subj_df["Condition"] == cond]
            if subset.empty:
                print(f"subset empty for {subj} {cond}")
                continue

            # --- build basis matrix from basis_codes for this subject ---
            basis_rows = []
            for code in basis_codes:
                row_basis = subj_df[subj_df["Condition"] == code]
                if row_basis.empty:
                    basis_rows = []
                    break
                basis_rows.append(row_basis.iloc[0])
            if len(basis_rows) != len(basis_codes):
                print(f"missing basis for {subj}")
                continue

            # stack basis + target rows into one small DataFrame (raw first)
            basis_df_raw = pd.DataFrame(basis_rows)
            target_row_raw = subset.iloc[0]

            if use_null_diff:
                # subtract null from basis and target
                basis_df = basis_df_raw[features].copy()
                for i in range(len(basis_df)):
                    basis_df.iloc[i] = basis_df_raw.iloc[i][features].to_numpy(dtype=float) - \
                                       null_row[features].to_numpy(dtype=float)

                target_row_vals = target_row_raw[features].to_numpy(dtype=float) - \
                                  null_row[features].to_numpy(dtype=float)
                target_row = pd.Series(target_row_vals, index=features)
            else:
                basis_df = basis_df_raw[features]
                target_row = target_row_raw[features]

            # build tmp for NaN filtering (basis + target)
            tmp = pd.concat(
                [basis_df, target_row.to_frame().T],
                axis=0
            )

            # columns that are non-NaN across all basis rows and this condition
            valid_features = tmp.columns[~tmp.isnull().any(axis=0)].tolist()

            if len(valid_features) == 0:
                print(f"no valid features for {subj} {cond}")
                continue

            # X: basis vectors, Y: target vector
            X = np.column_stack(
                [basis_df.iloc[i][valid_features].to_numpy(dtype=float)
                 for i in range(len(basis_codes))]
            )
            X_interacted = interaction_transformer.fit_transform(X)

            y = target_row[valid_features].to_numpy(dtype=float)

            model.fit(X, y)
            model_interacted.fit(X_interacted, y)

            y_pred = model.predict(X)
            y_pred_interacted = model_interacted.predict(X_interacted)

            r2 = r2_score(y, y_pred)
            r2_interacted = r2_score(y, y_pred_interacted)

            rmse = np.sqrt(np.mean((y - y_pred) ** 2))
            rmse_interacted = np.sqrt(np.mean((y - y_pred_interacted) ** 2))

            # assumes 3 basis codes and 3 main + 3 interaction terms
            w1, w2, w3 = model.coef_
            wi1, wi2, wi3, wi12, wi13, wi23 = model_interacted.coef_

            rows.append({
                "Subject": subj,
                "Condition": cond,
                "w_" + basis_codes[0]: w1,
                "w_" + basis_codes[1]: w2,
                "w_" + basis_codes[2]: w3,
                "RMSE": rmse,
                "R2": r2,
            })

            rows_interacted.append({
                "Subject": subj,
                "Condition": cond,
                "wi_s": wi1,
                "wi_a": wi2,
                "wi_b": wi3,
                "wi_sxa": wi12,
                "wi_sxb": wi13,
                "wi_axb": wi23,
                "RMSE": rmse_interacted,
                "R2": r2_interacted,
            })

            y_comparison.append({
                "Subject": subj,
                "Condition": cond,
                "Actual": [y],
                "Predicted": [y_pred],
                "Predicted_Interacted": [y_pred_interacted],
            })

    return rows, rows_interacted, y_comparison

## Helper functions

In [ ]:
def load_model_features(csv_path, comp_prefixes=None, max_components=None, zscore_within_subject=True):
    """
    Load a single subject_condition_*.csv file and return:
      final_df      : DataFrame with Subject, Condition, and selected component columns
      feature_cols  : list of component column names to use as features

    comp_prefixes: list of prefixes to keep (e.g. ["PC", "Comp", "NMF"]).
                   If None, use all non-id columns as features.
    max_components: if not None and component columns have trailing integers,
                    keep only those with index <= max_components.
    """
    df = pd.read_csv(csv_path)

    # Identify candidate feature columns
    id_cols = {"Subject", "Condition", "Speed", "Target", "Balance"}
    other_cols = [c for c in df.columns if c not in id_cols]

    if comp_prefixes is not None:
        comp_cols = [c for c in other_cols if any(c.startswith(p) for p in comp_prefixes)]
    else:
        comp_cols = other_cols

    if max_components is not None:
        selected = []
        for c in comp_cols:
            # extract trailing integer, if present
            suffix = "".join(ch for ch in c if ch.isdigit())
            if suffix.isdigit() and int(suffix) <= max_components:
                selected.append(c)
        comp_cols = sorted(selected)

    # standardize within subject if requested
    if zscore_within_subject and comp_cols:
        df[comp_cols] = df.groupby("Subject")[comp_cols].transform(
            lambda x: (x - x.mean()) / x.std()
        )

    # drop rows with all-NaN in component columns
    df = df.dropna(subset=comp_cols, how="all").reset_index(drop=True)

    return df, comp_cols
    
def get_digits(cond_label, n_digits=3):
    """
    Extract the first `n_digits` digits from a condition label like 's0a0b0' -> '000'.
    """
    digits = re.findall(r"\d", cond_label)
    if len(digits) < n_digits:
        raise ValueError(
            f"Could not find {n_digits} digits in condition label '{cond_label}'"
        )
    return "".join(digits[:n_digits])


def parse_sab(cond):
    """Split a condition label 'sXaYbZ' into its (speed, accuracy, balance) levels."""
    m = re.match(r"s(\d+)a(\d+)b(\d+)", cond)
    if m is None:
        return None, None, None
    return int(m.group(1)), int(m.group(2)), int(m.group(3))


def adjust_lightness(hex_color, factor):
    """
    adjust lightness of a hex color in HLS space
    factor > 1 -> lighter, factor < 1 -> darker.
    """
    r, g, b = mcolors.to_rgb(hex_color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = max(0, min(1, l * factor))
    r2, g2, b2 = colorsys.hls_to_rgb(h, l, s)
    return mcolors.to_hex((r2, g2, b2))


def compute_condition_subsets(df_data, BASIS, n_digits=3, verbose=False):
    """
    Given a dataframe `df_data` with column 'Condition' and an iterable `BASIS`
    of basis condition labels, compute:

      - BASIS_FULL: set of full labels in df_data whose digit code is in BASIS
      - COND_REP:   conditions whose 3 digits are each in the represented levels
                   (per position) induced by BASIS
      - COND_UNREP: conditions with at least one digit in an unrepresented level

    Parameters
    ----------
    df_data : pandas.DataFrame
        Must contain a column 'Condition' with string labels.
    BASIS : iterable of str
        Basis condition labels (full labels, not digit codes).
    n_digits : int, default 3
        Number of digits to extract from labels.
    verbose : bool, default False
        If True, prints summary info.

    Returns
    -------
    BASIS_FULL : set of str
    COND_REP : set of str
    COND_UNREP : set of str
    represented_levels : dict[int, set[int]]
        For each digit position, which levels are represented in the BASIS.
    label_to_digits : dict[str, str]
        Mapping from full condition label to its digit code.
    """

    # 1) Convert BASIS -> digit codes
    BASIS_FULL_LIST = list(BASIS)
    BASIS_DIGITS = {get_digits(lbl, n_digits=n_digits) for lbl in BASIS_FULL_LIST}

    # 2) Get represented levels per digit position from BASIS_DIGITS
    represented_levels = {pos: set() for pos in range(n_digits)}
    for dcode in BASIS_DIGITS:  # e.g. '120'
        if len(dcode) != n_digits:
            raise ValueError(f"Digit code '{dcode}' does not have length {n_digits}.")
        for pos in range(n_digits):
            represented_levels[pos].add(int(dcode[pos]))

    # 3) Get all conditions in dataset and map to digit codes
    all_conditions = np.unique(df_data["Condition"].values)
    label_to_digits = {c: get_digits(c, n_digits=n_digits) for c in all_conditions}

    # BASIS as full condition labels (subset that actually appear in df_data)
    BASIS_FULL = {c for c, d in label_to_digits.items() if d in BASIS_DIGITS}

    # 4) Global REP / UNREP sets (excluding BASIS_FULL)
    COND_REP   = set()  # all digits represented
    COND_UNREP = set()  # at least one digit unrepresented

    for c in all_conditions:
        if c in BASIS_FULL:
            continue  # never include basis in either subset

        dcode = label_to_digits[c]
        all_rep = True
        for pos in range(n_digits):
            d = int(dcode[pos])
            if d not in represented_levels[pos]:
                all_rep = False
                break

        if all_rep:
            COND_REP.add(c)
        else:
            COND_UNREP.add(c)

    if verbose:
        print("\nGlobal BASIS full condition labels:")
        print(sorted(BASIS_FULL))
        print("\nGlobal condition subsets (excluding BASIS_FULL):")
        print("  REP   (all digits represented):")
        print("   ", sorted(COND_REP))
        print("  UNREP (at least one digit unrepresented):")
        print("   ", sorted(COND_UNREP))

    return BASIS_FULL, COND_REP, COND_UNREP, represented_levels, label_to_digits

## Load raw spatiotemporal metrics and subjective ratings

`gm_raw_df` holds one row per subject x condition of measured gait metrics;
`combined_targets` holds the matching self-reported priorities, normalized to
sum to one across the three objectives.

In [ ]:
# --- paths and lists ---
base_path = DATA_DIR
bmh_files = [
    "data_BMH01.xlsx", "data_BMH02.xlsx", "data_BMH21.xlsx", "data_BMH06.xlsx",
    "data_BMH07.xlsx", "data_BMH08.xlsx", "data_BMH09.xlsx", "data_BMH10.xlsx",
    "data_BMH13.xlsx", "data_BMH19.xlsx", "data_BMH20.xlsx", "data_BMH17.xlsx",
]
filepath_survey = DATA_DIR / "Subjective_Responses.xlsx"
targets_columns = ["Balance", "Foot Placement", "Walking Speed"]

# --- helper functions ---

def normalize_rows_to_one(df, columns):
    normalized_df = df.copy()
    for column in columns:
        normalized_df[column] = normalized_df[column].astype(float)
    for index, row in df.iterrows():
        row_sum = row[columns].sum()
        for column in columns:
            normalized_df.at[index, column] = row[column] / row_sum
    return normalized_df

def read_and_adjust_sheet(filename, sheet_name):
    total_rows = 35
    rows_to_skip = [1, 2]
    nrows = total_rows - len(rows_to_skip) - 6
    df = pd.read_excel(filename, sheet_name=sheet_name,
                       skiprows=rows_to_skip, nrows=nrows)
    df.index = range(1, len(df) + 1)
    return df

# --- collect raw gait metrics and targets ---

combined_data = pd.DataFrame()
combined_targets = pd.DataFrame()

for bmh_file in bmh_files:
    data_path = os.path.join(base_path, bmh_file)
    df_data = pd.read_excel(data_path)

    walking_speed_mapping = {"Slow": 0, "Medium": 1, "Fast": 2}
    accuracy_mapping     = {"Low": 0, "Medium": 1, "High": 2}
    balance_mapping      = {"Low": 0, "Medium": 1, "High": 2}

    sheet_name = bmh_file.split("_")[1].replace(".xlsx", "")
    df_survey = read_and_adjust_sheet(filepath_survey, sheet_name)
    df_survey = normalize_rows_to_one(df_survey, targets_columns)

    targets = df_survey[targets_columns].copy()
    targets["Subject"] = sheet_name

    predictors_columns = [
        "Mean Error Straights", "Mean Width Straights (mm)", "Straights Width Variability (mm)",
        "Mean Length Straights (mm)", "Straights Length Variability (mm)",
        "Average Speed (m/s)", "EE", "Head Angle (deg)", "Condition",
    ]

    # map categorical to numeric, build Condition
    df_data["Walking Speed"] = df_data["Walking Speed"].map(walking_speed_mapping).astype(int)
    df_data["Accuracy"]      = df_data["Accuracy"].map(accuracy_mapping).astype(int)
    df_data["Balance"]       = df_data["Balance"].map(balance_mapping).astype(int)
    df_data["Condition"] = df_data.apply(
        lambda row: f"s{row['Walking Speed']:.0f}a{row['Accuracy']:.0f}b{row['Balance']:.0f}",
        axis=1,
    )

    # use "EE Wkg" instead of the raw "EE" column
    df_data["EE"] = df_data["EE Wkg"]

    predictors = df_data[predictors_columns].copy()
    predictors["Subject"] = sheet_name

    targets["Condition"] = df_data["Condition"].values

    combined_data    = pd.concat([combined_data, predictors], ignore_index=True)
    combined_targets = pd.concat([combined_targets, targets],   ignore_index=True)

# raw gait metrics dataframe keyed by Subject+Condition
gm_raw_cols = [
    "Mean Error Straights", "Mean Width Straights (mm)", "Straights Width Variability (mm)",
    "Mean Length Straights (mm)", "Straights Length Variability (mm)",
    "Average Speed (m/s)", "EE", "Head Angle (deg)",
]
gm_raw_df = combined_data[["Subject", "Condition"] + gm_raw_cols].copy()

# Centre each metric within subject (swap for a z-score if scale matters too).
gm_norm_df = gm_raw_df.copy()
gm_norm_df[gm_raw_cols] = gm_raw_df.groupby("Subject")[gm_raw_cols].transform(lambda x: x - x.mean())

## Configuration

`model_files` lists the feature sets to compare; each must already exist, having
been produced by `00_reference_extract_movement_features.ipynb`.

`GOF_SCOPE` selects which conditions the headline fit statistics summarize.
*REP* conditions are those whose prompt levels all appear somewhere in the basis
set; *UNREP* conditions require at least one level the basis never visits, and
are therefore the stricter generalization test.

In [ ]:
BASIS=("s2a0b0", "s1a2b0", "s1a0b2")
REGRESS_DIFF = True

# Feature sets to compare: label -> component-score CSV.
FEATURE_DIR = DATA_DIR
model_files = {
    "EMG-PCA-5": FEATURE_DIR / "decomp_emg_pca_fixed5_synergy.csv",
    "Kin-PCA-5": FEATURE_DIR / "decomp_kinematics_pca_fixed5_kin_synergy.csv",
    "SPT-PCA-5": FEATURE_DIR / "decomp_gait_metrics_pca_fixed5.csv",
}

# Component columns to use, and an optional cap on how many.
COMP_PREFIXES = "Comp"
MAX_COMPONENTS = None

# Neutral condition subtracted from basis and target when REGRESS_DIFF is True.
NULL_CODE = "s1a0b0"

# choose which subset to summarize:
#   "all_nonbasis" (all non-basis conditions)
#   "rep"          (COND_REP)
#   "unrep"        (COND_UNREP)
GOF_SCOPE = "unrep"

## Goodness of fit across feature sets

Fits the weight model once per feature set and summarizes fit quality over the
condition subset chosen by `GOF_SCOPE`.

In [ ]:
BASIS_FULL, COND_REP, COND_UNREP, represented_levels, label_to_digits = \
    compute_condition_subsets(combined_data, BASIS, n_digits=3, verbose=False)

def get_gof_mask(df_simple, scope):
    """
    Return boolean mask for rows used in GOF summary
    and a human-readable label.
    """
    if scope == "all_nonbasis":
        mask = ~df_simple["Condition"].isin(BASIS)
        label = "All non-basis"
    elif scope == "rep":
        mask = df_simple["Condition"].isin(COND_REP)
        label = "REP conditions"
    elif scope == "unrep":
        mask = df_simple["Condition"].isin(COND_UNREP)
        label = "UNREP conditions"
    else:
        raise ValueError(f"Unknown GOF_SCOPE: {scope}")
    return mask, label

weights_simple = {}
gof_summary = []

for model_name, path in model_files.items():
    # print(f"\nModel: {model_name}")
    df_model, feature_cols = load_model_features(
        path,
        comp_prefixes=COMP_PREFIXES,
        max_components=MAX_COMPONENTS,
        zscore_within_subject=False,  # PCA/NMF features already standardized
    )

    if not feature_cols:
        print("  No feature columns found; skipping")
        continue

    print(f"Using {len(feature_cols)} features from {path}")

    rows, rows_interacted, y_comp = estimate_weights(
        df_model,
        features=feature_cols,
        basis_codes=BASIS,
        use_null_diff=REGRESS_DIFF,
        skip_subjects=(),
    )

    df_simple = pd.DataFrame(rows)
    if df_simple.empty:
        print("  No valid subject/condition rows; skipping")
        continue

    # --- masks for GOF subsets ---
    mask_all_nonbasis = ~df_simple["Condition"].isin(BASIS)
    mask_rep          = df_simple["Condition"].isin(COND_REP)
    mask_unrep        = df_simple["Condition"].isin(COND_UNREP)

    # --- summary mask depending on GOF_SCOPE ---
    mask_summary, summary_label = get_gof_mask(df_simple, GOF_SCOPE)

    # mean GOF for chosen subset
    rmse_mean = df_simple.loc[mask_summary, "RMSE"].mean()
    r2_mean   = df_simple.loc[mask_summary, "R2"].mean()

    # mean R² for REP vs UNREP (always computed, independent of GOF_SCOPE)
    r2_rep   = df_simple.loc[mask_rep,   "R2"].mean() if mask_rep.any()   else np.nan
    r2_unrep = df_simple.loc[mask_unrep, "R2"].mean() if mask_unrep.any() else np.nan

    # per-subject R² for chosen subset
    subj_r2 = (
        df_simple.loc[mask_summary]
        .groupby("Subject")["R2"]
        .mean()
        .reset_index()
    )
    subj_r2["Model"] = model_name

    weights_simple[model_name] = {
        "df": df_simple,
        "subj_r2": subj_r2,
    }

    # flag modality by model name
    if "Kin" in model_name:
        modality = "Kinematics"
    elif "SPT" in model_name:
        modality = "Gait Metrics"
    elif "EMG" in model_name:
        modality = "EMG"
    else:
        modality = "Other"

    # Save summary info for this model
    gof_summary.append({
        "Model": model_name,
        "Modality": modality,
        "n_features": len(feature_cols),
        "RMSE_Mean": rmse_mean,
        "R2_Mean": r2_mean,    # for GOF_SCOPE subset
        "R2_Rep": r2_rep,      # across all REP
        "R2_Unrep": r2_unrep,  # across all UNREP
        "GOF_Scope": summary_label,
    })

# --- Collect all per-subject R² into a single DataFrame ---
subj_r2_df = (
    pd.concat([d["subj_r2"] for d in weights_simple.values()],
              ignore_index=True)
)

r2_stats_by_model = (
    subj_r2_df
    .groupby("Model")["R2"]
    .agg(R2_Min="min", R2_Max="max", R2_SD="std")
    .reset_index()
)

gof_summary_df = pd.DataFrame(gof_summary).merge(
    r2_stats_by_model,
    on="Model",
    how="left",
)

print("\n============== Goodness-of-fit summary "
      f"({summary_label} for R2_Mean / RMSE_Mean) ==============")
print(gof_summary_df.to_string(index=False, float_format="{:.3f}".format))

print("\nSubset sizes:")
print(f"  BASIS_FULL: {len(BASIS_FULL)}")
print(f"  COND_REP:   {len(COND_REP)}")
print(f"  COND_UNREP: {len(COND_UNREP)}")

## Diagnostic: are the basis conditions collinear?

The three basis conditions are the regressors. If their feature profiles are too
similar the estimated weights become unstable even when R² looks healthy.

In [ ]:
# Rebuilds the design matrix exactly as estimate_weights() does, including the
# null-condition subtraction when REGRESS_DIFF is on.
null_code_for_vif = NULL_CODE

vif_summary = []

for model_name, path in model_files.items():
    df_model, feature_cols = load_model_features(
        path,
        comp_prefixes=COMP_PREFIXES,
        max_components=MAX_COMPONENTS,
        zscore_within_subject=False,
    )
    if not feature_cols:
        continue

    basis_rows = []
    for subj in df_model["Subject"].unique():
        subj_df = df_model[df_model["Subject"] == subj]
        if not all(code in subj_df["Condition"].values for code in BASIS):
            continue

        cols = [
            subj_df[subj_df["Condition"] == code].iloc[0][feature_cols].to_numpy(dtype=float)
            for code in BASIS
        ]

        if REGRESS_DIFF:
            null_df = subj_df[subj_df["Condition"] == null_code_for_vif]
            if null_df.empty:
                continue
            null_vec = null_df.iloc[0][feature_cols].to_numpy(dtype=float)
            cols = [c - null_vec for c in cols]

        basis_rows.append(np.column_stack(cols))  # (n_features, 3) per subject

    if not basis_rows:
        print(f"{model_name}: no subjects with complete basis conditions, skipping")
        continue

    # Stack across subjects: rows = (subject, feature) pairs, columns = basis conditions
    X_basis = np.vstack(basis_rows)
    X_basis = X_basis[~np.isnan(X_basis).any(axis=1)]
    X_with_const = np.column_stack([np.ones(len(X_basis)), X_basis])

    for i, code in enumerate(["const"] + list(BASIS)):
        vif_summary.append({
            "Model": model_name,
            "basis_condition": code,
            "VIF": variance_inflation_factor(X_with_const, i),
        })

vif_df = pd.DataFrame(vif_summary)
vif_table = vif_df.pivot(index="basis_condition", columns="Model", values="VIF")
print(vif_table.to_string(float_format="{:.2f}".format))

## Goodness-of-fit figure and statistics

In [ ]:
modality_order = ["Gait Metrics", "Kinematics", "EMG"]

palette_bars = {
    "Gait Metrics": "#F5CFCE",
    "Kinematics":   "#FFE0BC",
    "EMG":          "#FBF0AB",
    "Other":        "gray",
}

palette_subjs = plt.cm.tab20

# ------ Set ordering ------ #
gof_summary_df = gof_summary_df.copy()

gof_summary_df["Modality"] = pd.Categorical(
    gof_summary_df["Modality"],
    categories=modality_order,
    ordered=True,
)

gof_summary_df = gof_summary_df.sort_values(
    by=["Modality", "Model"]
).reset_index(drop=True)

model_order = gof_summary_df["Model"].tolist()

# Map each Model -> Modality -> color
model_to_modality = (
    gof_summary_df
    .drop_duplicates("Model")
    .set_index("Model")["Modality"]
)

palette_models = {
    m: palette_bars.get(model_to_modality[m], "gray")
    for m in model_order
}

# Make subj df follow same model order
subj_r2_df = subj_r2_df.copy()
subj_r2_df["Model"] = pd.Categorical(
    subj_r2_df["Model"],
    categories=model_order,
    ordered=True,
)

plt.figure(figsize=(min(1 * len(model_files), 12), 4))

# ------ BOX PLOT (per‑subject R² for GOF_SCOPE subset) ------ #
ax = sns.boxplot(
    data=subj_r2_df,
    x="Model",
    y="R2",
    order=model_order,
    palette=palette_models,   # <-- use palette, not color
    showfliers=False,
    showmeans=True,
    meanprops={"marker": "x", "markeredgecolor": "black", "markersize": 6},
    width=0.7,
)

ax.set_ylim(0, 1)
ax.set_ylabel("R²", fontsize=14)
ax.set_xlabel("", fontsize=13)
# Tick labels follow model_order; update if model_files changes.
ax.set_xticklabels(["Spatiotemp", "Joint Kin", "EMG"])
ax.spines[['right', 'top']].set_visible(False)
plt.xticks(rotation=30, ha="right")
plt.suptitle("Goodness-of-fit", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Paired t-tests on per-subject R2 (same subjects across feature sets),
# Holm-corrected across the pairwise comparisons.

# Pivot to one row per subject, one column per model, so R2 values are aligned
# by subject before each pairwise comparison.
r2_wide = subj_r2_df.pivot(index="Subject", columns="Model", values="R2")

pairs = list(itertools.combinations(model_order, 2))

ttest_results = []
for m1, m2 in pairs:
    paired = r2_wide[[m1, m2]].dropna()  # keep only subjects present in both models

    if len(paired) < 2:
        # not enough matched subjects to run a paired test
        ttest_results.append({
            "Model1": m1,
            "Model2": m2,
            "n_subjects": len(paired),
            "t_stat": np.nan,
            "p_uncorrected": np.nan,
        })
        continue

    t_stat, p_val = stats.ttest_rel(paired[m1], paired[m2])

    ttest_results.append({
        "Model1": m1,
        "Model2": m2,
        "n_subjects": len(paired),
        "t_stat": t_stat,
        "p_uncorrected": p_val,
    })

ttest_df = pd.DataFrame(ttest_results)

# Holm correction (only over rows where a test was actually run)
valid = ttest_df["p_uncorrected"].notna()
reject, p_corr, _, _ = multipletests(
    ttest_df.loc[valid, "p_uncorrected"].values,
    alpha=0.05,
    method="holm"
)

ttest_df.loc[valid, "p_corrected"] = p_corr
ttest_df.loc[valid, "reject_H0"] = reject

print(ttest_df.sort_values("p_corrected"))

## Weights for the selected feature set

Takes the weights from one feature set forward for the remaining figures.
Weights are made non-negative and normalized to sum to one per row so they can
be read as a share of attention across the three objectives.

In [ ]:
selected_model = "SPT-PCA-5"   # key in model_files
NORMALIZE_WEIGHTS = True       # scale |weights| so each row sums to 1
weight_cols = [f"w_{code}" for code in BASIS]

# Get weights from selected dataframe
weights_df = weights_simple[selected_model]["df"].copy()
# Optional normalization of weights [0 1]
if NORMALIZE_WEIGHTS:
    weights_df[weight_cols] = weights_df[weight_cols].abs()                     # take absolute values
    row_sums = weights_df[weight_cols].sum(axis=1)                              # sum per row
    row_sums = row_sums.replace(0, np.nan)                                      # avoid division by zero if needed
    weights_df[weight_cols] = weights_df[weight_cols].div(row_sums, axis=0)     # normalize

# Export the weights for the selected feature set. data/weights_gait_metrics.xlsx
# is this table for selected_model = "SPT-PCA-5"; re-running overwrites it.
EXPORT_WEIGHTS = True
MODALITY_TAG = {"SPT-PCA-5": "gait_metrics", "Kin-PCA-5": "kinematics", "EMG-PCA-5": "emg"}

if EXPORT_WEIGHTS:
    weights_out = DATA_DIR / f"weights_{MODALITY_TAG.get(selected_model, selected_model)}.xlsx"
    weights_df.to_excel(weights_out, index=False)
    print(f"Wrote estimated weights to: {weights_out}")

## Dominant objective per condition

For each condition, the objective with the largest weight "wins", provided it
beats the runner-up by at least `eps`; otherwise the condition is left
unresolved. `main_winner` records the most common winner across participants and
the share of participants who agree.

In [ ]:
eps = 0.1  # dominance margin: top weight must beat the runner-up by this much to "win"
obj_cols = [f"w_{code}" for code in BASIS]

def winner_with_margin(row, cols=obj_cols, eps=eps):
    vals = row[cols]
    sorted_vals = vals.sort_values(ascending=False)
    best = sorted_vals.iloc[0]
    second = sorted_vals.iloc[1]
    if best - second < eps:
        return "No clear winner"
    else:
        best_col = sorted_vals.index[0]  # e.g., "w_s2a0b0"
        return best_col.replace("w_", "")

weights_df["winner_eps"] = weights_df.apply(winner_with_margin, axis=1)

# Summarize winners per condition
winner_summary = (
    weights_df
    .groupby(["Condition", "winner_eps"])
    .size()
    .reset_index(name="n")
)

# ensure 'prop' is a proportion
winner_summary["prop"] = (
    winner_summary["n"] /
    winner_summary.groupby("Condition")["n"].transform("sum")
)

# main_winner with s,a,b and prop
main_winner = (
    winner_summary
    .loc[winner_summary.groupby("Condition")["prop"].idxmax()]
    .rename(columns={"winner_eps": "main_winner"})
    .reset_index(drop=True)
)

main_winner[["s", "a", "b"]] = main_winner["Condition"].apply(
    lambda c: pd.Series(parse_sab(c))
)

# BASIS in a fixed order: speed, accuracy, balance.
basis_list = list(BASIS)
if len(basis_list) != 3:
    print(f"Warning: expected 3 BASIS entries, found {len(basis_list)}")

winner_colors_plotly = {
    basis_list[0]: "#4DA167",  # speed
    basis_list[1]: "#085BCF",  # accuracy
    basis_list[2]: "#662E9B",  # balance
    "No clear winner": "#777777",
}

print(f"main_winner computed for {len(main_winner)} conditions:")
print(main_winner[["Condition", "main_winner", "prop", "s", "a", "b"]].to_string(index=False))

## Dominant-objective cube

Each corner of the cube is one combination of prompt levels. Colour encodes the
dominant objective; lightness encodes how many participants agreed on it, ramped
in HLS space rather than by opacity so the contrast stays visible across the
range.

In [ ]:
PROP_MIN, PROP_MAX = 1 / 3, 1.0   # 1/3 = chance level with 3 objective categories
LIGHT_HI = 0.88                    # pale end of the ramp (prop at/near chance level)
LIGHT_LO_OFFSET = 0.06             # how much brighter than the "natural" hex lightness at prop=1.0

def prop_to_color(hex_color, prop):
    """Map a category's base hex + a participant-agreement proportion to an RGB color
    by interpolating lightness in HLS space (pale -> saturated), not by opacity."""
    r, g, b = mcolors.to_rgb(hex_color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    light_lo = min(1, l + LIGHT_LO_OFFSET)
    t = np.clip((prop - PROP_MIN) / (PROP_MAX - PROP_MIN), 0, 1)
    l_new = LIGHT_HI - t * (LIGHT_HI - light_lo)
    return colorsys.hls_to_rgb(h, l_new, s)

fig_mpl = plt.figure(figsize=(5, 10))
ax_mpl = fig_mpl.add_subplot(111, projection="3d")
ax_mpl.set_box_aspect([0.97, 1, 1])  # force a perfect cube: equal-length x/y/z axes regardless of figure shape

# transparent panes (no gray cube-wall background) + no default grid
for axis in (ax_mpl.xaxis, ax_mpl.yaxis, ax_mpl.zaxis):
    axis.pane.fill = False
    axis.pane.set_edgecolor((1, 1, 1, 0))
ax_mpl.grid(False)
fig_mpl.patch.set_alpha(0)
ax_mpl.patch.set_alpha(0)

# custom internal grid lines (kept, lighter than the default cube walls)
grid_vals = [0, 1, 2]
for v in grid_vals:
    for w in grid_vals:
        ax_mpl.plot([0, 2], [v, v], [w, w], color="lightgray", linewidth=0.5, alpha=0.6)
        ax_mpl.plot([v, v], [0, 2], [w, w], color="lightgray", linewidth=0.5, alpha=0.6)
        ax_mpl.plot([v, v], [w, w], [0, 2], color="lightgray", linewidth=0.5, alpha=0.6)

for _, row in main_winner.iterrows():
    color = prop_to_color(winner_colors_plotly[row["main_winner"]], row["prop"])
    # x = Speed, y = Accuracy, z = Balance -- matches the original axis layout
    # (Speed along the front-bottom edge, Accuracy along the front-right edge,
    # so Slow + Ignore meet at the front-right corner, as in the original figure)
    ax_mpl.scatter(row["s"], row["a"], row["b"], color=color,
                   s=160, edgecolors="white", linewidths=0.8, depthshade=False)

ax_mpl.set_xlabel("Speed"); ax_mpl.set_ylabel("Accuracy"); ax_mpl.set_zlabel("Balance")
ax_mpl.set_xticks([0, 1, 2]); ax_mpl.set_xticklabels(["Slow", "Typical", "Fast"])
ax_mpl.set_yticks([0, 1, 2]); ax_mpl.set_yticklabels(["Ignore", "Near", "Accurate"])
ax_mpl.set_zticks([0, 1, 2]); ax_mpl.set_zticklabels(["Zero", "Low", "High"])
ax_mpl.invert_xaxis()  # Fast on the left, Slow on the right (front-bottom edge)
ax_mpl.view_init(elev=15, azim=-60)  # adjust to taste to match poster crop

plt.tight_layout()
plt.show()

## Estimated vs. perceived weights

Compares the weights recovered from movement data with participants' own ratings
of what they were prioritizing, and with the raw gait metric most closely tied to
each objective.

In [ ]:
# Subjective ratings, ordered to match the basis order (speed, accuracy, balance).
subj_cols = ["Walking Speed", "Foot Placement", "Balance"]
sdf = combined_targets.copy()

# Exclude the basis conditions, as in the estimated weights.
if "Condition" in sdf.columns:
    mask_s = ~sdf["Condition"].isin(BASIS)
    sdf = sdf[mask_s].copy()
    
# ---------------------------------------------------
# 1. Map the three weight columns to objectives
# ---------------------------------------------------
# weight_cols is [speed, accuracy, balance], derived from BASIS above.
metrics_cols = [
    "Average Speed (m/s)",              # for speed
    "Mean Error Straights",             # for accuracy
    "Straights Width Variability (mm)", # for balance
]

# x-axis metric and plot titles for each DOF
pairings = [
    ("Average Speed (m/s)",              "Speed weight vs. Speed"),
    ("Mean Error Straights",             "Accuracy weight vs. FPE"),
    ("Straights Width Variability (mm)", "Balance weight vs. SWV"),
]

# Use raw or normalized gait metrics on x-axis
metrics_source = gm_raw_df   # or gm_norm_df

# Encoding from Condition = sXaYbZ
speed_colors   = {0: "#F2776D",   1: "#F5C11F", 2: "#1DBCC2"}  # s
balance_alphas = {0: 0.2,     1: 0.5,     2: 1}      # b
acc_markers    = {0: "o",     1: "^",     2: "s"}      # a

# ---------------------------------------------------
# 2. Condition-level averages
# ---------------------------------------------------
# gait metrics per Condition
metrics_cond = (
    metrics_source[["Condition"] + metrics_cols]
    .groupby("Condition", as_index=False)
    .mean()
)

# estimated weights per Condition (weights_df already normalized row-wise & BASIS-filtered)
w_cond = (
    weights_df[["Condition"] + weight_cols]
    .groupby("Condition", as_index=False)
    .mean()
)

# subjective weights per Condition
s_cond = (
    sdf[["Condition"] + subj_cols]
    .groupby("Condition", as_index=False)
    .mean()
)

# merge
est_df = metrics_cond.merge(w_cond, on="Condition", how="inner")
subj_df = metrics_cond.merge(s_cond, on="Condition", how="inner")

# ---------------------------------------------------
# 3. Generic plotting helper
# ---------------------------------------------------
def scatter_weights_vs_metrics(df, y_cols, which="Estimated"):
    """
    df: condition-level dataframe with 'Condition', metrics_cols, and y_cols
    y_cols: list of 3 columns [speed_y, acc_y, bal_y]
    which: 'Estimated' or 'Subjective'
    """
    fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharey=False)
    fig.suptitle(f"{which} weights vs. gait metrics -- {selected_model}", y=1.03)

    for i, ax in enumerate(axes):
        y_col = y_cols[i]
        x_col, base_title = pairings[i]

        if y_col not in df.columns:
            raise KeyError(
                f"{which} plot: column '{y_col}' not in df. "
                f"Available: {df.columns.tolist()}"
            )

        # ax.set_title(base_title.replace("weight", f"{which.lower()} weight"))
        ax.set_title(f"{which} {dof_titles[i].lower()} weight")

        # collect all x,y for regression (across conditions)
        x_all = []
        y_all = []

        # --- existing condition-style scatter ---
        for _, row in df.iterrows():
            cond = row["Condition"]
            s, a, b = parse_sab(cond)
            
            if s is None:
                continue

            color = speed_colors.get(s, "gray")
            alpha = balance_alphas.get(b, 0.3)
            marker = acc_markers.get(a, "o")

            x_val = row[x_col]
            y_val = row[y_col]

            ax.scatter(
                x_val,
                y_val,
                color=color,
                alpha=alpha,
                marker=marker,
                s=50,
            )

            x_all.append(x_val)
            y_all.append(y_val)

        x_all = np.asarray(x_all, dtype=float)
        y_all = np.asarray(y_all, dtype=float)

        # --- linear regression + R² over all points on this axis ---
        if len(x_all) >= 2 and np.nanstd(x_all) > 0:
            coeffs = np.polyfit(x_all, y_all, 1)  # [slope, intercept]
            slope, intercept = coeffs
            y_pred = np.polyval(coeffs, x_all)
            ss_res = np.sum((y_all - y_pred) ** 2)
            ss_tot = np.sum((y_all - np.nanmean(y_all)) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

            # regression line
            xx = np.linspace(np.nanmin(x_all), np.nanmax(x_all), 100)
            yy = slope * xx + intercept
            ax.plot(xx, yy, color="black", linestyle="--", linewidth=2)

            # print full-precision stats to console for manuscript reporting
            # (kept separate from the on-plot annotation, which stays at 2 decimals
            # for legibility)
            print(f"{which} | {base_title}")
            print(f"  y = {slope:.4f} * x + {intercept:.4f}")
            print(f"  R² = {r2:.4f}\n")

            # annotate R² on the plot
            ax.text(
                0.05, 0.95,
                f"R² = {r2:.2f}",
                transform=ax.transAxes,
                va="top", ha="left",
                fontsize=9,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.7)
            )

        ax.set_xlabel(x_col)
        ax.set_ylabel("Normalized weight")
        ax.set_ylim(0, 0.8)
    
    plt.tight_layout()
    plt.show()
    plt.close(fig)

# ---------------------------------------------------
# 4. direct correlation of subjective vs. estimated
# ---------------------------------------------------
# Merge estimated and subjective on Condition only (no metrics needed)
subj_est_df = w_cond.merge(s_cond, on="Condition", how="inner")

# Bring in the dominant-basis winner + proportion per condition, computed the
# same way as the Winner Cube plot, so marker styling can match it exactly.
subj_est_df = subj_est_df.merge(
    main_winner[["Condition", "main_winner", "prop"]],
    on="Condition", how="left",
)
subj_est_df["main_winner"] = subj_est_df["main_winner"].fillna("No clear winner")
subj_est_df["prop"] = subj_est_df["prop"].fillna(0.0)

# human-readable titles per DOF
dof_titles = ["Speed", "Accuracy", "Balance"]

def scatter_subjective_vs_estimated(df, est_cols, subj_cols):
    """
    df: dataframe with 'Condition', est_cols, subj_cols
    est_cols: list of 3 estimated weight cols (x-axis)
    subj_cols: list of 3 subjective weight cols (y-axis), matching DOF order
    """
    fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharey=False)
    fig.suptitle("Perceived vs. Estimated weights", y=1.03)

    for i, ax in enumerate(axes):
        ax.set_box_aspect(1)
        est_col = est_cols[i]
        subj_col = subj_cols[i]
        dof_name = dof_titles[i]

        if est_col not in df.columns or subj_col not in df.columns:
            raise KeyError(
                f"Missing columns for DOF {dof_name}: "
                f"needed {est_col} and {subj_col}, "
                f"have {df.columns.tolist()}"
            )

        ax.set_title(f"{dof_name} weight")

        x_all = []
        y_all = []

        for _, row in df.iterrows():
            cond = row["Condition"]
            s, a, b = parse_sab(cond)
            if s is None:
                continue

            # Style markers to match the Winner Cube plot exactly: color = dominant
            # basis, shading = proportion of subjects agreeing on that winner.
            # Uses the same prop_to_color() lightness ramp defined in the Winner
            # Cube (matplotlib) cell above -- NOT plain alpha-blending -- so this
            # plot and the cube/legend always render identical colors.
            winner_cat = row.get("main_winner", "No clear winner")
            winner_prop = row.get("prop", 0.0)
            base_hex = winner_colors_plotly.get(
                winner_cat, winner_colors_plotly["No clear winner"]
            )
            face_color = prop_to_color(base_hex, winner_prop)

            x_val = row[est_col]   # estimated on x-axis
            y_val = row[subj_col]  # subjective on y-axis

            ax.scatter(
                x_val,
                y_val,
                facecolor=face_color,
                edgecolor="none",
                linewidth=1,
                marker="o",
                s=50,
            )

            x_all.append(x_val)
            y_all.append(y_val)

        x_all = np.asarray(x_all, dtype=float)
        y_all = np.asarray(y_all, dtype=float)

        # add identity line
        ax.plot([0, 1], [0, 1], color="gray", linestyle=":", linewidth=1)

        # linear regression & R²
        if len(x_all) >= 2 and np.nanstd(x_all) > 0:
            coeffs = np.polyfit(x_all, y_all, 1)
            slope, intercept = coeffs
            y_pred = np.polyval(coeffs, x_all)
            ss_res = np.sum((y_all - y_pred) ** 2)
            ss_tot = np.sum((y_all - np.nanmean(y_all)) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

            # regression line
            xx = np.linspace(np.nanmin(x_all), np.nanmax(x_all), 100)
            yy = slope * xx + intercept
            ax.plot(xx, yy, color="black", linestyle="--", linewidth=2)

            # print full-precision stats to console for manuscript reporting
            # (kept separate from the on-plot annotation, which stays at 2 decimals
            # for legibility)
            print(f"{dof_name} weight")
            print(f"  y = {slope:.4f} * x + {intercept:.4f}")
            print(f"  R² = {r2:.4f}\n")

            # annotate equation and R² on the plot
            ax.text(
                0.05, 0.95,
                f"y = {slope:.2f}x + {intercept:.2f}\nR² = {r2:.2f}",
                transform=ax.transAxes,
                va="top", ha="left",
                fontsize=9,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.7)
            )

        ax.set_xlabel("Estimated (normalized)")
        ax.set_ylabel("Perceived (normalized)")
        ax.set_xlim(0, 0.8)
        ax.set_ylim(0, 0.8)

    plt.tight_layout()
    plt.show()
    plt.close(fig)


# ---------------------------------------------------
# 5. Make plots
# ---------------------------------------------------
# Estimated (use your 3 DOF weight columns)
est_y_cols = weight_cols  # [w_s2a0b0, w_s1a2b0, w_s1a0b2]
scatter_weights_vs_metrics(est_df, est_y_cols, which="Estimated")

# Subjective (use the 3 subjective columns)
subj_y_cols = ["Walking Speed", "Foot Placement", "Balance"]
scatter_weights_vs_metrics(subj_df, subj_y_cols, which="Subjective")

scatter_subjective_vs_estimated(subj_est_df, weight_cols, subj_cols)

## Cost landscapes

Interpolated contours of each outcome cost over a plane defined by two execution
variables, showing how the measured costs trade off across the task space.

In [ ]:
# Execution variables (x, y)
x_feat = "Mean Length Straights (mm)"
y_feat = "Head Angle (deg)"

use_gaussian_smoothing = True  # Set to False to disable
gaussian_sigma = 3.0           # Increase for more smoothing (try 1.0-5.0)

In [ ]:
# Absolute costs.
# ----------------------------------------------------
# 0. Make a proper Time Cost column (1 / speed)
# ----------------------------------------------------
# Assume gm_raw_df["Average Speed (m/s)"] exists
gm_raw_df = gm_raw_df.copy()
speed = gm_raw_df["Average Speed (m/s)"].to_numpy()

# mask out non-positive speeds to avoid division by zero/negatives
valid_speed_mask = np.isfinite(speed) & (speed > 0)
time_cost = np.full_like(speed, np.nan, dtype=float)
time_cost[valid_speed_mask] = 1.0 / speed[valid_speed_mask]

gm_raw_df["Time Cost (s/m)"] = time_cost

# ----------------------------------------------------
# 1. Setup outcomes for color
# ----------------------------------------------------
fields = {
    "Time Cost":            "Time Cost (s/m)",
    "AP Error":             "Mean Error Straights",
    "ML Variability":       "Straights Width Variability (mm)",
    "Energy Cost":          "EE",
}

# ----------------------------------------------------
# 2. Pull data from gm_raw_df
# ----------------------------------------------------
x = gm_raw_df[x_feat].to_numpy()
y = gm_raw_df[y_feat].to_numpy()

Z_all = np.column_stack([
    gm_raw_df[col].to_numpy() for col in fields.values()
])

# keep only rows that are finite for x, y and all outcomes
mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(Z_all).all(axis=1)
x = x[mask]
y = y[mask]
Z_all = Z_all[mask, :]

# ----------------------------------------------------
# 3. Build grid in (step length, head angle) space
# ----------------------------------------------------
n_grid = 150  # increase for smoother contours, decrease if slow
xi = np.linspace(x.min(), x.max(), n_grid)
yi = np.linspace(y.min(), y.max(), n_grid)
XI, YI = np.meshgrid(xi, yi)

# ----------------------------------------------------
# 4. Make 2×2 panel of smooth contour plots
# ----------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(24, 6), sharex=True, sharey=True)
axes = axes.ravel()

for ax, (panel_title, colname), z_vals in zip(axes, fields.items(), Z_all.T):
    # interpolate onto grid
    ZI = griddata(
        points=np.c_[x, y],
        values=z_vals,
        xi=(XI, YI),
        method="linear"
    )

     # ===== APPLY GAUSSIAN SMOOTHING =====
    if use_gaussian_smoothing:
        # Create a mask for NaN values before smoothing
        nan_mask = np.isnan(ZI)
        # Replace NaNs temporarily with interpolated values for smoothing
        ZI_for_smooth = ZI.copy()
        if np.any(nan_mask):
            ZI_for_smooth[nan_mask] = np.nanmean(ZI)
        # Apply Gaussian filter
        ZI = gaussian_filter(ZI_for_smooth, sigma=gaussian_sigma)
        # Restore NaNs in the original locations (optional)
        ZI[nan_mask] = np.nan
    # ====================================

    # filled contours (masking NaNs automatically)
    cf = ax.contourf(
        XI, YI, ZI,
        levels=20,
        cmap="Spectral_r",
        alpha=0.9
    )

    # raw points colored by their actual (unsmoothed) value
    # use the same colormap + normalization as the contours
    ax.scatter(
        x, y,
        c=z_vals,
        cmap="Spectral_r",
        norm=cf.norm,          # matches contour color scale
        s=15,
        edgecolor="k",
        linewidth=0.3,
        alpha=0.8
    )

    ax.set_xlabel("Execution Variable 1")
    ax.set_title(panel_title)

    # ---- horizontal colorbar at bottom of this panel ----
    cbar = fig.colorbar(
        cf,
        ax=ax,
        orientation="horizontal",
        pad=0.1,     # small vertical gap so plots aren't squished
        shrink=1.0,   # full length under the axis
        aspect=30     # long + thin bar
    )
    cbar.set_label(colname)
    # ensure ticks/label appear at bottom
    cbar.ax.xaxis.set_ticks_position("bottom")
    cbar.ax.xaxis.set_label_position("bottom")
    

axes[0].set_ylabel("Execution Variable 2")

plt.show()